# Supervised Fine-Tuning: build it, prove the mask, watch the loss drop

Hand a raw **base** language model `What is the capital of France?` and it may just keep generating *more questions* — it only ever learned to continue internet text. It *knows* the answer is Paris (that fact is in its weights from pretraining); it was simply never taught that an instruction is a request for an **answer**. **Supervised Fine-Tuning (SFT)** fixes this: continue the *same* next-token training, but on curated **(prompt, response)** demonstrations, with the loss **masked to the response**.

This notebook builds SFT from scratch and **proves** the one idea that makes it work — the **prompt mask** (labels `-100` on prompt tokens, so the loss is computed only on the response). We will:
1. build a tiny GPT-style decoder LM (real next-token cross-entropy, no library),
2. format (instruction, response) pairs with a minimal **chat template**,
3. build labels with `-100` on the prompt (**completion-only** masking),
4. prove numerically that the masked loss is **exactly** the response-only average and **differs** from loss-on-all-tokens,
5. run a few SFT steps and watch the response-token loss collapse.

It runs on CPU in a few seconds. This is the same verified code as the companion `supervised_fine_tuning.py`.

## The problem: a base model *completes*, it doesn't *answer*

A pretrained base LM's only skill is **next-token prediction** — continue text the way the web would. That is not the same skill as *being a helpful assistant*: recognizing an instruction and producing a well-formatted answer that **stops** when done. SFT teaches that **behavior** (not new facts) from worked examples. Everything below is the mechanism that does it.

> **Step 1 — imports, seed, and honest device reporting.** We pin everything to **CPU** so the printed loss trace is reproducible regardless of the accelerator this runs on, and we print the detected device honestly (matching the repo's device-agnostic convention).

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

IGNORE_INDEX = -100   # PyTorch cross_entropy skips any target == this
D_MODEL, N_HEADS, N_LAYERS, MAX_LEN = 64, 4, 2, 64
SEED, LEARNING_RATE, N_SFT_STEPS = 0, 3e-3, 60

# Detect the best accelerator, but PIN to CPU for a reproducible trace (device honesty).
DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
train_device = "cpu"
torch.manual_seed(SEED)
print(f"device: {train_device} (detected {DEVICE}; pinned to CPU for reproducibility)")
print("torch:", torch.__version__)

device: cpu (detected mps; pinned to CPU for reproducibility)
torch: 2.12.0


> **Step 2 — the chat template.** A demonstration pair becomes **one token stream** wrapped in special tokens (`<s>`/`</s>`) and role tags (`<user>`/`<assistant>`). Real templates (Llama-3, ChatML) use longer tokens, but the **structure is identical**: specials delimit turns, role tags say who speaks, and an end token says where to stop. We split into `prompt` (given to the model) and `full` (prompt + response the loss is computed on).

In [2]:
BOS, USER_TAG, ASSISTANT_TAG, EOS = "<s>", "<user>", "<assistant>", "</s>"

DEMOS = [
    ("translate hello to french", "bonjour"),
    ("capital of france", "paris"),
    ("opposite of hot", "cold"),
    ("two plus two", "four"),
]

def format_chat(instruction, response):
    """Return (prompt_text, full_text). prompt = what the model is GIVEN; full = prompt + response."""
    prompt = f"{BOS} {USER_TAG} {instruction} {ASSISTANT_TAG}"
    full = f"{prompt} {response} {EOS}"
    return prompt, full

p, f = format_chat(*DEMOS[0])
print("prompt:", p)
print("full  :", f)

prompt: <s> <user> translate hello to french <assistant>
full  : <s> <user> translate hello to french <assistant> bonjour </s>


> **Step 3 — a tiny word-level tokenizer.** We build a deterministic vocabulary over the chat-formatted corpus (sorted, so token ids are reproducible). `encode` turns text into a 1-D LongTensor of ids. Small and readable — the point is the masking, not the tokenizer.

In [3]:
def build_tokenizer(demos):
    tokens = {BOS, USER_TAG, ASSISTANT_TAG, EOS}
    for instruction, response in demos:
        _, full = format_chat(instruction, response)
        tokens.update(full.split())
    stoi = {tok: i for i, tok in enumerate(sorted(tokens))}  # sorted -> deterministic ids
    itos = {i: tok for tok, i in stoi.items()}
    return stoi, itos

def encode(text, stoi, device):
    return torch.tensor([stoi[t] for t in text.split()], dtype=torch.long, device=device)

stoi, itos = build_tokenizer(DEMOS)
vocab_size = len(stoi)
print(f"toy vocab size: {vocab_size} tokens")

toy vocab size: 19 tokens


> **Step 4 — build the labels with the prompt mask (the heart of SFT).** `labels` is a copy of `input_ids` with the first `n_prompt` positions overwritten by `-100`. Those positions contribute **zero loss and zero gradient**, so the model is trained only to produce the **response**, never to re-emit the prompt it was handed. Print the per-token mask and read it by eye — this single table is what SFT *is*.

In [4]:
def build_example(instruction, response, stoi, device):
    """Return (input_ids, labels, n_prompt). labels = input_ids with the prompt set to -100."""
    prompt_text, full_text = format_chat(instruction, response)
    input_ids = encode(full_text, stoi, device)
    n_prompt = len(prompt_text.split())          # leading tokens that are the prompt
    labels = input_ids.clone()
    labels[:n_prompt] = IGNORE_INDEX             # MASK the prompt -> loss on response only
    return input_ids, labels, n_prompt

input_ids, labels, n_prompt = build_example(*DEMOS[0], stoi, train_device)
print(f"{'pos':>4} | {'token':<14} | {'label':>6} | trained?")
print("-" * 44)
for pos, tok_id in enumerate(input_ids.tolist()):
    label = labels[pos].item()
    trained = "no (prompt)" if label == IGNORE_INDEX else "YES (response)"
    print(f"{pos:>4} | {itos[tok_id]:<14} | {label:>6} | {trained}")

 pos | token          |  label | trained?
--------------------------------------------
   0 | <s>            |   -100 | no (prompt)
   1 | <user>         |   -100 | no (prompt)
   2 | translate      |   -100 | no (prompt)
   3 | hello          |   -100 | no (prompt)
   4 | to             |   -100 | no (prompt)
   5 | french         |   -100 | no (prompt)
   6 | <assistant>    |   -100 | no (prompt)
   7 | bonjour        |      4 | YES (response)
   8 | </s>           |      0 | YES (response)


> **Step 5 — a tiny GPT-style decoder LM.** Token + learned positional embeddings → a stack of transformer blocks fed a **causal mask** (so position *i* sees only ≤ *i*) → a linear head to vocab logits. Small enough to train on CPU in seconds, but the loss path (causal self-attention + next-token cross-entropy) is the **real thing**, so the SFT demonstration is faithful.

In [5]:
class TinyCausalLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, D_MODEL)
        self.pos_emb = nn.Embedding(MAX_LEN, D_MODEL)
        layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEADS, dim_feedforward=4 * D_MODEL,
            batch_first=True, activation="gelu",
        )
        self.blocks = nn.TransformerEncoder(layer, num_layers=N_LAYERS)
        self.ln_f = nn.LayerNorm(D_MODEL)
        self.head = nn.Linear(D_MODEL, vocab_size)

    def forward(self, input_ids):                      # (B, T) -> (B, T, V)
        seq_len = input_ids.shape[1]
        positions = torch.arange(seq_len, device=input_ids.device)
        x = self.token_emb(input_ids) + self.pos_emb(positions)
        causal = torch.triu(torch.ones(seq_len, seq_len, device=input_ids.device,
                                       dtype=torch.bool), diagonal=1)  # True = blocked
        x = self.blocks(x, mask=causal)
        return self.head(self.ln_f(x))

model = TinyCausalLM(vocab_size).to(train_device)
logits = model(input_ids.unsqueeze(0))
print("logits shape (B, T, V):", tuple(logits.shape))

logits shape (B, T, V): (1, 9, 19)


> **Step 6 — the masked next-token cross-entropy.** The teacher-forcing shift: predict token *t+1* from position *t*, so slice `logits[:, :-1]` and `labels[:, 1:]`. `ignore_index=-100` makes every prompt position contribute nothing. Set the labels to the real ids everywhere instead, and you recover ordinary LM loss — **SFT is pretraining's loss with a 0/1 mask on the prompt.**

In [6]:
def causal_lm_loss(logits, labels, ignore_index=IGNORE_INDEX):
    shift_logits = logits[:, :-1, :].contiguous()   # drop last position (no t+1 target)
    shift_labels = labels[:, 1:].contiguous()       # drop first label (no t-1 input)
    return F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),  # (B*(T-1), V)
        shift_labels.view(-1),                         # (B*(T-1),)
        ignore_index=ignore_index,                     # -100 -> zero loss & grad
    )

model.eval()
with torch.no_grad():
    logits = model(input_ids.unsqueeze(0))
    loss_masked = causal_lm_loss(logits, labels.unsqueeze(0))
print(f"masked (response-only) loss: {loss_masked.item():.4f}")

masked (response-only) loss: 3.3993


> **Step 7 — PROVE the mask is response-only (assertions before any training).** We re-derive the response-only loss **by hand without `ignore_index`** — compute per-token cross-entropy across the whole sequence, then average **only** the response positions. If this equals the masked loss, the mask is provably skipping the prompt (not approximating it). We also confirm it **differs** from loss-on-all-tokens — otherwise masking would be a no-op.

In [7]:
def response_only_loss_by_hand(logits, input_ids, n_prompt):
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = input_ids[:, 1:].contiguous()        # UN-masked real ids everywhere
    per_token = F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1), reduction="none",        # one loss per position
    )
    seq_minus_1 = shift_labels.size(1)
    target_positions = torch.arange(1, seq_minus_1 + 1, device=logits.device)
    response_mask = target_positions >= n_prompt        # original index >= n_prompt -> response
    return per_token[response_mask].mean()

with torch.no_grad():
    loss_all  = causal_lm_loss(logits, input_ids.unsqueeze(0))   # trains on prompt too (WRONG)
    loss_hand = response_only_loss_by_hand(logits, input_ids.unsqueeze(0), n_prompt)

print(f"loss on ALL tokens         : {loss_all.item():.4f}  (the WRONG objective)")
print(f"loss on RESPONSE only (-100): {loss_masked.item():.4f}  (SFT objective)")
print(f"same, re-derived by hand    : {loss_hand.item():.4f}  (should match masked)")

assert torch.allclose(loss_masked, loss_hand, atol=1e-6), "mask is not response-only!"
assert not torch.allclose(loss_masked, loss_all, atol=1e-3), "masking changed nothing?!"
print("assert OK: masked loss == response-only average, and != loss-on-all-tokens")

loss on ALL tokens         : 2.8253  (the WRONG objective)
loss on RESPONSE only (-100): 3.3993  (SFT objective)
same, re-derived by hand    : 3.3993  (should match masked)
assert OK: masked loss == response-only average, and != loss-on-all-tokens


## Now run SFT: watch the response-token loss collapse

Correctness of the mask is settled, so we can train. We batch the four demos (padding labels with `-100`, so padding is free), then run AdamW for a few steps and print the **response-token loss** dropping. This is SFT learning the demonstrated behavior.

In [8]:
# Re-seed and build a FRESH model right before training so this trace is IDENTICAL to
# supervised_fine_tuning.py and make_figures_13.py (dropout in the forward passes above
# advances the global RNG; re-seeding here makes the training init RNG-identical everywhere).
torch.manual_seed(SEED)
model = TinyCausalLM(vocab_size).to(train_device)
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
examples = [build_example(i, r, stoi, train_device) for i, r in DEMOS]
max_len = max(ex[0].size(0) for ex in examples)
pad_id = stoi[EOS]   # SAME pad_id as the .py and the figure generator (cross-file consistency)
batch_inputs = torch.full((len(examples), max_len), pad_id, dtype=torch.long, device=train_device)
batch_labels = torch.full((len(examples), max_len), IGNORE_INDEX, dtype=torch.long, device=train_device)
for row, (ids, labs, _) in enumerate(examples):
    batch_inputs[row, : ids.size(0)] = ids
    batch_labels[row, : labs.size(0)] = labs

# Note: step-0 here (~2.92) is a DIFFERENT measurement from the eval masked loss above (3.3993):
# this is the 4-example PADDED batch under train() (dropout active); that was a SINGLE example
# under eval()/no_grad. They are not meant to match -- different data, different mode.
print(f"{'step':>5} | {'response-loss':>13}")
print("-" * 23)
for step in range(N_SFT_STEPS + 1):
    logits = model(batch_inputs)
    loss = causal_lm_loss(logits, batch_labels)   # masked: response tokens only
    if step % 10 == 0:
        print(f"{step:>5} | {loss.item():>13.4f}")
    if step == N_SFT_STEPS:
        break
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

 step | response-loss
-----------------------
    0 |        2.9196
   10 |        0.0894
   20 |        0.0296
   30 |        0.0126
   40 |        0.0073
   50 |        0.0056


   60 |        0.0045


> **It learned the behavior.** Feed the *prompt only* for an unseen-format check and read off the model's next token. Before SFT it would continue like the web; after SFT it completes the instruction with the demonstrated response.

In [9]:
model.eval()
instruction, gold = DEMOS[1]   # "capital of france" -> "paris"
prompt_text, _ = format_chat(instruction, gold)
prompt_ids = encode(prompt_text, stoi, train_device).unsqueeze(0)
with torch.no_grad():
    next_logits = model(prompt_ids)[0, -1]
    predicted = itos[int(next_logits.argmax())]
print(f"prompt '{instruction}' -> model's next token: '{predicted}' (gold: '{gold}')")

prompt 'capital of france' -> model's next token: 'paris' (gold: 'paris')


## What we saw

- **The prompt mask is the whole trick** — labels `-100` on prompt tokens, so the loss is computed **only** on the response. We *proved* it: the masked loss equals a hand-rolled response-only average and differs from loss-on-all-tokens.
- **SFT reuses pretraining's loss** — next-token cross-entropy, unchanged; only *which tokens count* changes.
- **It teaches behavior, not facts** — the model learned to *answer* the demonstrated prompts; on a real base model the knowledge is already there and SFT surfaces it in the right **format**.
- **Watch overfitting** — our loss collapsed to ~0 because the set is tiny; real SFT is **1-3 epochs** on curated data, and driving the loss to 0 on real data *is* the overfitting failure mode.
- **One trace, four places** — the training trace printed above (`2.9196 → 0.0045`) is byte-identical to `supervised_fine_tuning.py`, the figure caption, and the page, because all three re-seed and build a fresh model right before the loop.

**Hands-on experiment:** comment out the re-seed + `model = TinyCausalLM(...)` line in the training cell and re-run from the top. Without the fresh, re-seeded model the loop trains the *eval-mode* model that already saw forward passes — the printed step-0 loss shifts, and your trace stops matching the `.py`. That tiny demonstration is exactly the reproducibility lesson: in PyTorch, **where you place `manual_seed` relative to every RNG-consuming op (dropout, init) decides whether two runs agree** — get it wrong and "identical" code produces different numbers.

**Where next:** [LoRA & PEFT](../../12-LoRA-and-PEFT/12-LoRA-and-PEFT.md) (run this on one GPU), [Instruction Tuning](../../14-Instruction-Tuning/14-Instruction-Tuning.md) (SFT at task-scale), [RLHF & DPO](../../15-RLHF-and-DPO/15-RLHF-and-DPO.md) (preference tuning on top of the SFT model).